> ## SOLUTION / ANSWER KEY &mdash; Lab 7.4
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-04-extract-fields.ipynb`](../lab-04-extract-fields.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.4 &mdash; Extract: Mess In, Structure Out

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Pull defined fields out of an unstructured email
- Use a closed set of intents; handle a missing order id
- Return the tight schema the rest of the pipeline consumes

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Extract — mess in, structure out](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = "/tmp/biaa-lab-07-04"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
**Extract** turns unstructured input into structured data (deck slide 10): an email *"my order from
last Tuesday still hasn't arrived, ref 4471, getting frustrated"* becomes
`{"order_id": "4471", "intent": "delivery", "sentiment": "negative"}`. Keys: a **tight schema** (only
the fields you'll use, intents from a **closed set**), **handle missing data** (return `None`, don't
invent an id), and **type consistency** &mdash; `order_id` is a **string** because the orders DB is
keyed by strings. Extract is usually the **first step** in the chain &mdash; extract &rarr; route
&rarr; draft. (A keyword extractor is deterministic and auditable; a model can extract too, but you
must still validate its output against a closed schema.)

In [ ]:
sample = "Hi, my order from last Tuesday still hasn't arrived, ref 4471, getting frustrated."
print("unstructured in:", sample)
print("we want out    : {order_id, intent, sentiment}")

## Build it
Complete `extract`: pull the order id (digits), classify the intent from a closed set, and read a
rough sentiment.

In [ ]:
INTENTS = ("refund", "delivery", "cancel", "other")

def extract(email):
    text = email.lower()
    # order id: the digits in the message, or None if there are none
    digits = "".join(ch for ch in email if ch.isdigit())
    order_id = digits if digits else None
    # intent: map keywords to a label from the CLOSED set INTENTS
    if "refund" in text or "money back" in text:
        intent = "refund"
    elif any(w in text for w in ("deliver", "arrive", "late", "where is", "hasn't")):
        intent = "delivery"
    elif "cancel" in text:
        intent = "cancel"
    else:
        intent = "other"
    # sentiment: negative if the customer sounds unhappy
    sentiment = "negative" if any(w in text for w in ("frustrated", "angry", "late", "still", "unhappy")) else "neutral"
    return {"order_id": order_id, "intent": intent, "sentiment": sentiment}

try:
    print(extract("my order 4471 still hasn't arrived, getting frustrated"))
    print(extract("please cancel my subscription"))
    print(extract("I want a refund"))
    print(extract("where is my stuff?"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The order id comes out a **string** (`"4471"`), matching the DB key; a message with no digits yields `None`, not a made-up id.
- `intent` is always one of the **closed set** &mdash; that's what makes the label safe to branch on in the router.
- The keyword rules mis-read some phrasings (e.g. "never showed up") &mdash; a real, visible failure mode you'll see again in the capstone.

## Your turn (open task &mdash; no grader)
Feed `extract` a few of your own emails &mdash; especially awkward ones like *"my parcel never showed up"*
(no keyword hit) &mdash; and see where it mislabels. **What good looks like:** you can name exactly which
phrasings slip past the keywords, which tells you where a model-based extractor would earn its keep.

---
### Key takeaway
Extract is the workhorse: it turns email, chat and forms into rows your systems can process. A tight schema plus missing-data handling is what makes it reliable enough to build on.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>